# Лабораторная 10. Финальный debug case

Цель: разобрать плохой Spark job и объяснить, что именно не так.

In [1]:
from pathlib import Path
import shutil
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import broadcast

spark = (SparkSession.builder.appName('lab-10-final-debug-case').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.driver.maxResultSize', '512m')
    .config('spark.sql.shuffle.partitions', '200')
    .config('spark.sql.adaptive.enabled', 'false')
    .config('spark.sql.autoBroadcastJoinThreshold', '-1')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base = Path('spark_core_data').absolute()
base_uri = base.as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
customers = spark.read.parquet(f'{base_uri}/customers')
order_items = spark.read.parquet(f'{base_uri}/order_items')
products = spark.read.parquet(f'{base_uri}/products')
print('Spark UI:', spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/09 07:52:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Spark UI: http://f8a3a643ec8b:4040


## Плохой job
В коде намеренно есть проблемы: AQE выключен, 200 shuffle partitions для маленьких данных, broadcast отключён, несколько join, groupBy и `coalesce(1)` перед записью.

In [2]:
bad = (
    orders
    .join(customers, 'customer_id')
    .join(order_items, 'order_id')
    .join(products, 'product_id')
    .groupBy('region', 'category', 'status')
    .agg(
        F.countDistinct('order_id').alias('orders_cnt'),
        F.sum(F.col('quantity') * F.col('item_price')).alias('revenue')
    )
    .orderBy(F.desc('revenue'))
)
bad.explain('formatted')

== Physical Plan ==
* Sort (38)
+- Exchange (37)
   +- * HashAggregate (36)
      +- Exchange (35)
         +- * HashAggregate (34)
            +- * HashAggregate (33)
               +- Exchange (32)
                  +- * HashAggregate (31)
                     +- * Project (30)
                        +- * SortMergeJoin Inner (29)
                           :- * Sort (23)
                           :  +- Exchange (22)
                           :     +- * Project (21)
                           :        +- * SortMergeJoin Inner (20)
                           :           :- * Sort (14)
                           :           :  +- Exchange (13)
                           :           :     +- * Project (12)
                           :           :        +- * SortMergeJoin Inner (11)
                           :           :           :- * Sort (5)
                           :           :           :  +- Exchange (4)
                           :           :           :     +- * Filter (

In [3]:
out = base / 'final_bad_output'
if out.exists():
    shutil.rmtree(out)
bad.coalesce(1).write.mode('overwrite').parquet(out.as_uri())
out.as_uri()

'file:///materials/seminar_04_spark_core/practice/spark_core_data/final_bad_output'

## Диагностика
Откройте Spark UI и ответьте:

- Где появился лишний shuffle? лишние shuffle появляются на всех этапах Exchange (4, 9, 13, 18, 22, 27, 32, 35, 37)
- Какой join strategy выбрал Spark? SortMergeJoin
- Что делает `coalesce(1)` перед записью? собирает все на одной ноде
- Почему 200 shuffle partitions плохо на этих данных? потому что они небольшие, shuffle read макисмум 8.4 mбайт и расходы на планирование превышают пользу от параллелизации
- Какой stage самый дорогой? Почему?  первая полная агрегация (HashAggregate) после всех join
- Где в плане видны `Exchange`, `SortMergeJoin`, `Sort`, `HashAggregate`? `Exchange` и `Sort` для всех датафреймов, затем `SortMergeJoin` их всех и между ними  - `Exchange` и `Sort` промежуточных результатов и затем (`HashAggregate`,`Exchange`, `HashAggregate`) 2 раза и `Exchange` с `Sort`

## Улучшенный вариант
Запустите после диагностики. Важно не просто выполнить код, а объяснить каждое изменение.

In [4]:
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.shuffle.partitions', '16')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '20m')

better = (
    orders.select('order_id', 'customer_id', 'status')
    .join(broadcast(customers.select('customer_id', 'region')), 'customer_id')
    .join(order_items.select('order_id', 'product_id', 'quantity', 'item_price'), 'order_id')
    .join(broadcast(products.select('product_id', 'category')), 'product_id')
    .groupBy('region', 'category', 'status')
    .agg(
        F.countDistinct('order_id').alias('orders_cnt'),
        F.sum(F.col('quantity') * F.col('item_price')).alias('revenue')
    )
)
better.explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (24)
+- HashAggregate (23)
   +- Exchange (22)
      +- HashAggregate (21)
         +- HashAggregate (20)
            +- Exchange (19)
               +- HashAggregate (18)
                  +- Project (17)
                     +- BroadcastHashJoin Inner BuildRight (16)
                        :- Project (12)
                        :  +- BroadcastHashJoin Inner BuildRight (11)
                        :     :- Project (7)
                        :     :  +- BroadcastHashJoin Inner BuildRight (6)
                        :     :     :- Filter (2)
                        :     :     :  +- Scan parquet  (1)
                        :     :     +- BroadcastExchange (5)
                        :     :        +- Filter (4)
                        :     :           +- Scan parquet  (3)
                        :     +- BroadcastExchange (10)
                        :        +- Filter (9)
                        :           +- Scan parquet  (8)
               

In [5]:
out2 = base / 'final_better_output'
if out2.exists():
    shutil.rmtree(out2)
better.repartition(4).write.mode('overwrite').parquet(out2.as_uri())
out2.as_uri()

'file:///materials/seminar_04_spark_core/practice/spark_core_data/final_better_output'

## Мини-отчёт

```text
Что было плохо:
неоптимальный linegae
Нет broadcast


Как я это увидел:
очень много ненужных шагов - аж 38, 
много exchange 
долгое выполнение (2.9 min)
Используется SortMergeJoin
coalesce(1)

Что я изменил:
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
выбор только нужных колонок
 .config('spark.sql.autoBroadcastJoinThreshold', '-1') > spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '20m')
bad.coalesce(1) > better.repartition(4)

Почему это должно помочь:
даст возможность оптимизировать lineage через AQE
разрешает сливать партии при необходимости 
уменьшить объемы данных
Маленькие таблицы (customers, products) рассылаются на рабочие узлы и не нужен лишний shuffle, объединени происходит локально
будет параллелльность
```

Ожидаемые направления улучшения:

- включить AQE;
- уменьшить shuffle partitions для маленьких данных;
- broadcast маленькие справочники;
- заменить `coalesce(1)` на разумное количество partitions;
- выбирать только нужные колонки до join;
- объяснить каждое изменение через Spark UI и `explain`.

In [6]:
spark.stop()